# Step 6 — Consume the deployed YOLO endpoint

The KServe `InferenceService` from notebook 05 is live. We'll POST each of the 10 held-out images in [`datasets/ppe/images/test/`](datasets/ppe/images/test/) — images the VLM never saw and the YOLO model was never trained on — and render the predicted boxes.

This closes the loop: raw images → VLM annotations → trained YOLO → registered + deployed on RHOAI → callable over HTTPS from any production workload that needs real-time PPE detection.

In [1]:
%%capture
!pip install requests pillow matplotlib

In [2]:
import base64
import json
import mimetypes
import os
from pathlib import Path

import matplotlib.pyplot as plt
import requests
from PIL import Image, ImageDraw, ImageFont

In [3]:
# Prefer the endpoint notebook 05 wrote to disk; fall back to an env var.

ENDPOINT = os.environ.get("PPE_INFERENCE_ENDPOINT", "http://ppe-yolov8n-vlm-2-predictor.ppe-detection-cv.svc.cluster.local")

AUTH_TOKEN = os.environ.get("PPE_INFERENCE_TOKEN")
MODEL_NAME = os.environ.get("PPE_MODEL_NAME", "ppe-yolov8n-vlm-2")
CLASSES    = ["person", "helmet", "no-helmet", "vest", "no-vest", "gloves", "no-gloves"]
LABEL_COLORS = {
    "person":     (70,  130, 200),
    "helmet":     (0,   200, 0),
    "no-helmet":  (220, 20,  60),
    "vest":       (50,  205, 50),
    "no-vest":    (255, 69,  0),
    "gloves":     (32,  178, 170),
    "no-gloves":  (199, 21,  133),
}

print("Endpoint:", ENDPOINT)

Endpoint: http://ppe-yolov8n-vlm-2-predictor.ppe-detection-cv.svc.cluster.local


## Call the endpoint

Most Ultralytics-on-KServe runtimes accept a base64-encoded image in a KServe v2 `infer` payload. Adjust `build_payload` if your runtime expects a different schema.

In [4]:
def encode_image(path: Path) -> str:
    mime = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    b64 = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{b64}"


def auth_headers() -> dict:
    return {"Authorization": f"Bearer {AUTH_TOKEN}"} if AUTH_TOKEN else {}


def build_payload(image_path: Path) -> dict:
    return {
        "inputs": [
            {
                "name": "image",
                "shape": [1],
                "datatype": "BYTES",
                "data": [encode_image(image_path)],
            }
        ]
    }


def predict(image_path: Path) -> dict:
    url = f"{ENDPOINT.rstrip('/')}/v2/models/{MODEL_NAME}/infer"
    r = requests.post(
        url,
        json=build_payload(image_path),
        headers={"Content-Type": "application/json", **auth_headers()},
        timeout=120,
        verify=True,
    )
    r.raise_for_status()
    return r.json()

In [5]:
TEST_DIR = Path("datasets/ppe/images/test")
test_images = sorted(p for p in TEST_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"Held-out test images: {len(test_images)}")

# Smoke test on the first image.
response = predict(test_images[0])
print(json.dumps(response, indent=2)[:600])

Held-out test images: 10


ConnectionError: HTTPConnectionPool(host='ppe-yolov8n-vlm-2-predictor.ppe-detection-cv.svc.cluster.local', port=80): Max retries exceeded with url: /v2/models/ppe-yolov8n-vlm-2/infer (Caused by NewConnectionError("HTTPConnection(host='ppe-yolov8n-vlm-2-predictor.ppe-detection-cv.svc.cluster.local', port=80): Failed to establish a new connection: [Errno 111] Connection refused"))

## Normalize the response to `{label, bbox_2d, confidence}`

The Ultralytics runtime typically returns a JSON blob with `boxes: [{x1, y1, x2, y2, class, confidence}]` inside `outputs[0].data[0]`. Tweak this one function if your runtime uses a different schema.

In [ ]:
def normalize(response: dict) -> list:
    outputs = response.get("outputs", [])
    if not outputs:
        return []
    payload = outputs[0].get("data", [None])[0]
    if isinstance(payload, str):
        payload = json.loads(payload)
    detections = []
    for b in payload.get("boxes", payload.get("detections", [])):
        label = b.get("class") or b.get("label")
        if isinstance(label, int) and label < len(CLASSES):
            label = CLASSES[label]
        detections.append({
            "label": label,
            "bbox_2d": [b["x1"], b["y1"], b["x2"], b["y2"]],
            "confidence": b.get("confidence", b.get("score", 0.0)),
        })
    return detections


detections = normalize(response)
print(f"Got {len(detections)} detections on {test_images[0].name}.")
for d in detections[:10]:
    print(" ", d)

## Run inference on all 10 test images and show a grid

In [ ]:
def draw(image_path: Path, detections: list) -> Image.Image:
    img = Image.open(image_path).convert("RGB")
    d = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 14)
    except OSError:
        font = ImageFont.load_default()
    for det in detections:
        label = det["label"]
        x1, y1, x2, y2 = det["bbox_2d"]
        color = LABEL_COLORS.get(label, (0, 255, 0))
        d.rectangle([x1, y1, x2, y2], outline=color, width=3)
        text = f"{label} {det['confidence']:.2f}"
        tb = d.textbbox((0, 0), text, font=font)
        th = tb[3] - tb[1]
        d.rectangle([x1, max(0, y1 - th - 4), x1 + (tb[2] - tb[0]) + 6, y1], fill=color)
        d.text((x1 + 3, max(0, y1 - th - 2)), text, fill=(0, 0, 0), font=font)
    return img


OUTPUT_DIR = Path("output"); OUTPUT_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 5, figsize=(24, 10))
for ax, img_path in zip(axes.flat, test_images):
    dets = normalize(predict(img_path))
    annotated = draw(img_path, dets)
    annotated.save(OUTPUT_DIR / f"{img_path.stem}_deployed.jpg")
    ax.imshow(annotated)
    ax.set_title(f"{img_path.name} \u2014 {len(dets)} dets", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

---

**Recap of the workshop story.**
1. [01] Raw, unlabeled images → loaded into FiftyOne.
2. [02] Qwen2.5-VL drew bounding boxes on them, zero-shot — no training data, no manual labeling.
3. [03] Exported to YOLO format.
4. [04] Fine-tuned a small, fast YOLOv8n.
5. [05] Registered in RHOAI Model Registry, deployed with KServe.
6. [06] Called the deployed endpoint from this notebook — this is what a production app would do.